# Energy Landscape Visualizer — Training (Colab, free GPU)

Trains the contrastive energy-based model end to end and saves the weights to `exports/energy_model.pt`. Built entirely on free tooling: PyTorch on Colab's free GPU tier, synthetic data generated in process, no external dataset and no paid compute.

**Before running:** set the runtime to GPU via `Runtime > Change runtime type > Hardware accelerator > GPU`. The free tier typically allocates an NVIDIA T4. The notebook still runs on CPU (same code path, just slower) if no GPU is available.

## 1. Get the code into the session
Clone the repository (edit the URL to your fork), then move into it.

In [ ]:
!git clone https://github.com/<your-user>/ebm-energy-landscape.git
%cd ebm-energy-landscape

## 2. Install pinned dependencies
Colab ships most of these; installing from the pinned file guarantees reproducibility.

In [ ]:
!pip install -q -r requirements.txt

## 3. Confirm the GPU is live
This must report `CUDA available: True` and name the device on a GPU runtime.

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 4. Train end to end and save the weights
Uses the same `production_config` and `train` as `python training/train.py`, with the device set to the GPU when present. The checkpoint bundles the weights, the architecture kwargs needed to rebuild the network, the full training history, and the final metrics.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "training")
from train import production_config, train, load_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = Path("exports/energy_model.pt")
config = production_config(checkpoint_path, device)

model, history = train(config)

## 5. Verify the saved weights reload faithfully
A freshly reloaded model must reproduce the trained one on a probe batch, and the energy separation must hold.

In [ ]:
reloaded, payload = load_checkpoint(checkpoint_path, map_location=device)

torch.manual_seed(12345)
probe_maps = torch.randn(4, model.init_kwargs["map_channels"], config.resolution, config.resolution, device=device)
probe_paths = torch.randn(4, config.n_points, 2, device=device)
model.eval()
with torch.no_grad():
    delta = (model(probe_maps, probe_paths) - reloaded(probe_maps, probe_paths)).abs().max()

print("final accuracy :", round(payload["metrics"]["accuracy"], 3))
print("final gap      :", round(payload["metrics"]["gap"], 3))
print("reload max diff :", float(delta))
assert float(delta) < 1e-5, "reloaded weights do not match the trained model"
print("checkpoint saved to", checkpoint_path)

## 6. Download the checkpoint
Pull the weights file out of the Colab session so it can feed the Phase 4 sampler and the Phase 5 export layer.

In [ ]:
from google.colab import files
files.download(str(checkpoint_path))